<a href="https://colab.research.google.com/github/umairism/Porable-Ai-Agent/blob/main/Google_Colab_Portable_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Portable AI Agent - Google Colab Edition

[![Python 3.8+](https://img.shields.io/badge/python-3.8+-blue.svg)](https://www.python.org/downloads/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Offline](https://img.shields.io/badge/Offline-Ready-green.svg)](https://github.com/umairism/Porable-Ai-Agent)

Welcome to your **Portable AI Agent** running on Google Colab! This notebook allows you to run your self-contained, offline-capable AI agent with self-learning capabilities in the cloud.

## ✨ Features Available in Colab:
- 🧠 **Self-Learning AI**: Continuously adapts from interactions
- 💾 **Session Memory**: Remembers context during the session
- 📚 **Knowledge Base**: Local vector database
- 🖥️ **Interactive Interface**: Web-based chat interface
- ⚡ **GPU Acceleration**: Leverage Colab's free GPUs
- 🔄 **Real-time Learning**: Learns from every conversation

---

## 🚀 Quick Start Guide:
1. **Run Section 1**: Clone repository and install dependencies
2. **Run Section 2**: Initialize the AI Agent
3. **Run Section 3**: Start the web interface
4. **Run Section 4**: Start chatting with your AI!

---

## 📦 Section 1: Setup and Installation

This section will:
- Clone your Portable AI Agent repository
- Install all required dependencies
- Configure the environment for Colab

In [ ]:
# Clone the Portable AI Agent repository
import os
import sys
from pathlib import Path

print("🚀 Setting up Portable AI Agent on Google Colab...")
print("=" * 50)

# Clone the repository if it doesn't exist
if not os.path.exists('/content/Porable-Ai-Agent'):
    print("📥 Cloning repository...")
    !git clone https://github.com/umairism/Porable-Ai-Agent.git /content/Porable-Ai-Agent
else:
    print("✅ Repository already exists")

# Change to the project directory
os.chdir('/content/Porable-Ai-Agent')
print(f"📁 Current directory: {os.getcwd()}")

# Add project to Python path
if '/content/Porable-Ai-Agent' not in sys.path:
    sys.path.insert(0, '/content/Porable-Ai-Agent')
    print("✅ Added project to Python path")

In [ ]:
# Install dependencies with Colab optimizations
print("📦 Installing dependencies...")
print("=" * 50)

# Install PyTorch with CUDA support for faster processing
print("🔥 Installing PyTorch with CUDA support...")
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install core AI dependencies
print("\n🧠 Installing AI/ML libraries...")
!pip install transformers>=4.30.0 sentence-transformers>=2.2.0
!pip install numpy scipy scikit-learn
!pip install faiss-cpu  # Use CPU version for Colab compatibility

# Install web framework and utilities
print("\n🌐 Installing web framework and utilities...")
!pip install flask nltk tqdm psutil
!pip install pandas joblib requests pyyaml
!pip install cryptography

# Download required NLTK data
print("\n📚 Downloading NLTK data...")
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("\n✅ All dependencies installed successfully!")

## 🧠 Section 2: Initialize AI Agent

This section will:
- Check system resources
- Initialize the AI components
- Load or create the knowledge base
- Set up memory systems

In [ ]:
# Check system resources and GPU availability
import torch
import psutil
import os

print("🔍 System Information")
print("=" * 50)

# Check GPU
if torch.cuda.is_available():
    print(f"🚀 GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("🖥️ Running on CPU")

# Check RAM
ram = psutil.virtual_memory()
print(f"🧠 Available RAM: {ram.available / 1e9:.1f} GB / {ram.total / 1e9:.1f} GB")

# Check disk space
disk = psutil.disk_usage('/')
print(f"💽 Available Disk: {disk.free / 1e9:.1f} GB / {disk.total / 1e9:.1f} GB")

print("\n✅ System check completed!")

In [ ]:
# Initialize AI Agent components
print("🤖 Initializing Portable AI Agent...")
print("=" * 50)

try:
    # Import AI components
    from core.ai_engine import SelfLearningCore
    from knowledge.knowledge_base import KnowledgeBase
    from memory.conversation_memory import ConversationMemory
    
    print("📚 Initializing Knowledge Base...")
    knowledge_base = KnowledgeBase()
    
    print("💭 Initializing Conversation Memory...")
    conversation_memory = ConversationMemory()
    
    print("🧠 Initializing AI Engine...")
    ai_engine = SelfLearningCore(
        knowledge_base=knowledge_base,
        memory=conversation_memory
    )
    
    print("\n✅ AI Agent initialized successfully!")
    print("🎯 Ready for conversations and learning!")
    
except Exception as e:
    print(f"❌ Error initializing AI Agent: {e}")
    print("\n🔧 Trying alternative initialization...")
    
    # Fallback initialization
    class SimpleAI:
        def __init__(self):
            self.memory = []
            
        def chat(self, message):
            response = f"I received your message: '{message}'. I'm learning from this interaction!"
            self.memory.append((message, response))
            return response
    
    ai_engine = SimpleAI()
    print("✅ Fallback AI initialized!")

## 🌐 Section 3: Start Web Interface

This section will:
- Set up Flask web server for Colab
- Configure tunneling for external access
- Start the interactive chat interface

In [ ]:
# Install and setup tunneling for Colab
print("🌐 Setting up web interface for Colab...")
print("=" * 50)

# Install pyngrok for tunneling
!pip install pyngrok flask-ngrok

print("✅ Tunneling tools installed!")

In [ ]:
# Create Colab-optimized web interface
from flask import Flask, render_template_string, request, jsonify
from pyngrok import ngrok
import threading
import time

print("🚀 Creating web interface...")

# Create Flask app
app = Flask(__name__)
app.secret_key = 'colab-portable-ai-agent'

# HTML template for the chat interface
HTML_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>🤖 Portable AI Agent - Colab Edition</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            height: 100vh;
            display: flex;
            flex-direction: column;
        }
        .header {
            background: rgba(255,255,255,0.1);
            padding: 20px;
            text-align: center;
            color: white;
            backdrop-filter: blur(10px);
        }
        .chat-container {
            flex: 1;
            display: flex;
            max-width: 1200px;
            margin: 20px auto;
            background: white;
            border-radius: 15px;
            box-shadow: 0 15px 35px rgba(0,0,0,0.2);
            overflow: hidden;
        }
        .chat-messages {
            flex: 1;
            padding: 20px;
            overflow-y: auto;
            max-height: 600px;
        }
        .message {
            margin-bottom: 15px;
            padding: 12px 18px;
            border-radius: 18px;
            max-width: 80%;
        }
        .user-message {
            background: #007bff;
            color: white;
            margin-left: auto;
        }
        .ai-message {
            background: #f8f9fa;
            color: #333;
            border: 1px solid #e9ecef;
        }
        .input-container {
            padding: 20px;
            border-top: 1px solid #eee;
            display: flex;
            gap: 10px;
        }
        #messageInput {
            flex: 1;
            padding: 12px 18px;
            border: 1px solid #ddd;
            border-radius: 25px;
            outline: none;
            font-size: 16px;
        }
        #sendButton {
            padding: 12px 24px;
            background: #007bff;
            color: white;
            border: none;
            border-radius: 25px;
            cursor: pointer;
            font-size: 16px;
        }
        #sendButton:hover { background: #0056b3; }
        .status {
            text-align: center;
            color: #666;
            padding: 10px;
            font-style: italic;
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>🤖 Portable AI Agent</h1>
        <p>Self-Learning • Offline-Capable • Privacy-First</p>
        <p><strong>Running on Google Colab</strong></p>
    </div>
    
    <div class="chat-container">
        <div class="chat-messages" id="chatMessages">
            <div class="ai-message message">
                <strong>🤖 AI Agent:</strong> Hello! I'm your Portable AI Agent running on Google Colab. I can learn from our conversations and help you with various tasks. What would you like to talk about?
            </div>
        </div>
    </div>
    
    <div class="input-container">
        <input type="text" id="messageInput" placeholder="Type your message here..." autocomplete="off">
        <button id="sendButton">Send 🚀</button>
    </div>
    
    <div class="status" id="status">
        Status: Ready • Memory: Active • Learning: Enabled
    </div>

    <script>
        const chatMessages = document.getElementById('chatMessages');
        const messageInput = document.getElementById('messageInput');
        const sendButton = document.getElementById('sendButton');
        const status = document.getElementById('status');

        function addMessage(content, isUser) {
            const messageDiv = document.createElement('div');
            messageDiv.className = `message ${isUser ? 'user-message' : 'ai-message'}`;
            messageDiv.innerHTML = `<strong>${isUser ? '👤 You' : '🤖 AI Agent'}:</strong> ${content}`;
            chatMessages.appendChild(messageDiv);
            chatMessages.scrollTop = chatMessages.scrollHeight;
        }

        function sendMessage() {
            const message = messageInput.value.trim();
            if (!message) return;

            addMessage(message, true);
            messageInput.value = '';
            sendButton.disabled = true;
            sendButton.textContent = 'Thinking...';
            status.textContent = 'Status: AI is processing and learning...';

            fetch('/chat', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ message: message })
            })
            .then(response => response.json())
            .then(data => {
                addMessage(data.response, false);
                sendButton.disabled = false;
                sendButton.textContent = 'Send 🚀';
                status.textContent = `Status: Ready • Memory: ${data.memory_size || 'Active'} • Learning: Enabled`;
            })
            .catch(error => {
                console.error('Error:', error);
                addMessage('Sorry, there was an error processing your message.', false);
                sendButton.disabled = false;
                sendButton.textContent = 'Send 🚀';
                status.textContent = 'Status: Error occurred';
            });
        }

        sendButton.addEventListener('click', sendMessage);
        messageInput.addEventListener('keypress', function(e) {
            if (e.key === 'Enter') sendMessage();
        });

        // Focus on input
        messageInput.focus();
    </script>
</body>
</html>
'''

# Routes
@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route('/chat', methods=['POST'])
def chat():
    try:
        data = request.get_json()
        message = data.get('message', '')
        
        # Process message with AI
        if hasattr(ai_engine, 'chat'):
            response = ai_engine.chat(message)
        elif hasattr(ai_engine, 'generate_response'):
            response = ai_engine.generate_response(message)
        else:
            response = f"I received: '{message}'. I'm learning from this interaction!"
        
        memory_size = len(getattr(ai_engine, 'memory', [])) if hasattr(ai_engine, 'memory') else 'Active'
        
        return jsonify({
            'response': response,
            'memory_size': memory_size
        })
    except Exception as e:
        return jsonify({
            'response': f"Error processing message: {str(e)}",
            'memory_size': 0
        })

print("✅ Web interface created!")
print("🌐 Starting server...")

In [ ]:
# Start the web server with ngrok tunneling
import time
from threading import Thread

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# Start Flask in a separate thread
print("🚀 Starting Flask server...")
flask_thread = Thread(target=run_flask, daemon=True)
flask_thread.start()

# Wait for Flask to start
time.sleep(3)

# Create ngrok tunnel
print("🌐 Creating public URL with ngrok...")
public_url = ngrok.connect(5000)

print("\n" + "=" * 80)
print("🎉 SUCCESS! Your Portable AI Agent is now running!")
print("=" * 80)
print(f"🌐 Public URL: {public_url}")
print("\n🔗 Click the link above to access your AI Agent!")
print("\n💡 Features available:")
print("   • 🧠 Real-time learning from conversations")
print("   • 💾 Session memory (remembers context)")
print("   • 🚀 GPU-accelerated processing (if available)")
print("   • 🔒 Secure tunneled connection")
print("\n⚠️ Note: Keep this cell running to maintain the server!")
print("=" * 80)

## 💬 Section 4: Interactive Chat (Alternative)

If the web interface doesn't work, you can use this direct chat interface within Colab:

In [ ]:
# Direct chat interface for Colab
print("💬 Direct Chat Interface")
print("=" * 50)
print("🤖 AI Agent: Hello! I'm ready to chat and learn from our conversation.")
print("📝 Type your messages below. Type 'quit' to exit.\n")

conversation_history = []

while True:
    try:
        # Get user input
        user_message = input("👤 You: ")
        
        if user_message.lower() in ['quit', 'exit', 'bye']:
            print("🤖 AI Agent: Goodbye! Thanks for the conversation. I learned a lot!")
            break
        
        # Process with AI
        if hasattr(ai_engine, 'chat'):
            response = ai_engine.chat(user_message)
        elif hasattr(ai_engine, 'generate_response'):
            response = ai_engine.generate_response(user_message)
        else:
            # Simple response generation
            conversation_history.append(user_message)
            response = f"I understand you said: '{user_message}'. I'm learning from our conversation! (Memory: {len(conversation_history)} messages)"
        
        print(f"🤖 AI Agent: {response}")
        print()
        
    except KeyboardInterrupt:
        print("\n🤖 AI Agent: Chat interrupted. Thanks for the conversation!")
        break
    except Exception as e:
        print(f"❌ Error: {e}")
        continue

## 🔧 Section 5: Advanced Features & Management

Advanced features and system management tools:

In [ ]:
# System status and memory management
import psutil
import os

print("📊 System Status & Memory Management")
print("=" * 50)

# Memory usage
process = psutil.Process()
memory_info = process.memory_info()
print(f"🧠 Current Memory Usage: {memory_info.rss / 1024 / 1024:.2f} MB")

# GPU status
if torch.cuda.is_available():
    print(f"🚀 GPU Memory Used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"🚀 GPU Memory Cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# AI Agent status
if 'ai_engine' in locals():
    if hasattr(ai_engine, 'memory'):
        print(f"💭 Conversation Memory: {len(ai_engine.memory)} interactions")
    if hasattr(ai_engine, 'knowledge_base'):
        print(f"📚 Knowledge Base: Active")
    print(f"🤖 AI Engine: {type(ai_engine).__name__}")

print("\n🔄 Memory Management Options:")
print("   • Clear GPU cache: torch.cuda.empty_cache()")
print("   • Reset conversation: ai_engine.memory = []")
print("   • System cleanup: gc.collect()")

In [ ]:
# Save and backup current session
import pickle
import json
from datetime import datetime

print("💾 Session Management")
print("=" * 50)

# Create backup directory
backup_dir = '/content/ai_agent_backup'
os.makedirs(backup_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

try:
    # Save conversation memory
    if 'ai_engine' in locals() and hasattr(ai_engine, 'memory'):
        memory_file = f"{backup_dir}/conversation_memory_{timestamp}.pkl"
        with open(memory_file, 'wb') as f:
            pickle.dump(ai_engine.memory, f)
        print(f"💭 Conversation memory saved: {memory_file}")
    
    # Save session info
    session_info = {
        'timestamp': timestamp,
        'system': {
            'gpu_available': torch.cuda.is_available(),
            'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            'python_version': sys.version,
        },
        'ai_status': {
            'engine_type': type(ai_engine).__name__ if 'ai_engine' in locals() else None,
            'memory_size': len(ai_engine.memory) if hasattr(ai_engine, 'memory') else 0,
        }
    }
    
    session_file = f"{backup_dir}/session_info_{timestamp}.json"
    with open(session_file, 'w') as f:
        json.dump(session_info, f, indent=2)
    print(f"📋 Session info saved: {session_file}")
    
    print(f"\n✅ Session backup completed!")
    print(f"📁 Backup location: {backup_dir}")
    
except Exception as e:
    print(f"❌ Error saving session: {e}")

# List backup files
print(f"\n📂 Current backups:")
for file in os.listdir(backup_dir):
    print(f"   • {file}")

## 🔧 Troubleshooting & Tips

### Common Issues:
1. **Memory errors**: Restart runtime and try again
2. **GPU not available**: Check Runtime > Change runtime type > GPU
3. **Import errors**: Ensure all dependencies are installed
4. **Web interface not loading**: Try the direct chat interface instead

### Performance Tips:
- Enable GPU for faster processing
- Clear memory periodically: `torch.cuda.empty_cache()`
- Save important conversations using the backup feature
- Monitor system resources in Section 5

### Privacy & Security:
- All processing happens in your Colab session
- No data is sent to external servers (except through ngrok tunnel)
- Session data is temporary and cleared when runtime stops
- Use backup feature to save important conversations

---

## 🎉 Enjoy your Portable AI Agent on Google Colab!

**Created by**: Umair Malik  
**Repository**: [Porable-Ai-Agent](https://github.com/umairism/Porable-Ai-Agent)  
**License**: MIT

**Happy chatting and learning! 🤖✨**